---
# 02. 텍스트 전처리 및 감성점수 산출

## 분석 개요

버거+샌드위치+패스트푸드 업체의 리뷰 텍스트를 전처리하고
TF-IDF 기반 감성점수와 VADER 감성점수를 산출한다.

| 단계 | 내용 |
|---|---|
| STEP 2-1 | 데이터 로드 |
| STEP 2-2 | 텍스트 전처리 |
| STEP 2-3 | 긍정/부정 코퍼스 구성 |
| STEP 2-4 | TF-IDF 기반 감성 사전 구축 |
| STEP 2-5 | 리뷰별 감성점수 산출 |
| STEP 2-6 | 업체별 평균 감성점수 집계 |
| STEP 2-7 | VADER 검증 |
| STEP 2-8 | 저장 |

**입력 데이터**
- `biz_target_burger.csv`: 업체 정보
- `review_target_burger.csv`: 리뷰 데이터

**출력 데이터**
- `review_burger_cleaned.csv`: 전처리된 리뷰
- `biz_sentiment_burger.csv`: 업체별 TF-IDF 감성점수 + VADER 점수

## 공통 라이브러리 및 설정

In [1]:
import pandas as pd
import numpy as np
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
from sklearn.feature_extraction.text import TfidfVectorizer
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
import warnings
warnings.filterwarnings('ignore')

nltk.download('stopwords', quiet=True)

PATH_to_data = "../results"
PATH_to_save = "../results"

print("라이브러리 로드 완료")

라이브러리 로드 완료


---
## STEP 2-1. 데이터 로드

In [2]:
biz_target  = pd.read_csv(f"{PATH_to_data}/biz_target_burger.csv")
review_target = pd.read_csv(f"{PATH_to_data}/review_target_burger.csv")

print(f"업체 수: {len(biz_target):,}개")
print(f"리뷰 수: {len(review_target):,}개")
print(f"\n프랜차이즈: {biz_target['is_franchise'].sum():,}개")
print(f"독립 브랜드: {(~biz_target['is_franchise']).sum():,}개")

업체 수: 1,579개
리뷰 수: 178,094개

프랜차이즈: 804개
독립 브랜드: 775개


---
## STEP 2-2. 텍스트 전처리

restaurant과 동일한 전처리 방식을 적용한다.
- 소문자화, 숫자/구두점 제거
- 불용어 제거 (NLTK + 커스텀 불용어)
- 어간 추출 (Porter Stemmer)

전처리 결과는 런타임 끊김 대비 즉시 저장한다.

In [3]:
stop_words = set(stopwords.words('english')) | {"does", "not", "thing"}

MANUAL_STOPWORDS = {
    'al', 'also', 'alway', 'anoth', 'area', 'around', 'ask',
    'back', 'bite', 'box',
    'come', 'could', 'came',
    'dont', 'day', 'de', 'didnt',
    'even', 'ever', 'el',
    'get', 'give', 'got',
    'im', 'ive',
    'let', 'la', 'last',
    'make', 'made', 'mayb',
    'name', 'one', 'round',
    'someth', 'still', 'seem', 'sinc', 'sub', 'said',
    'told', 'that', 'think', 'two', 'though', 'thought', 'took',
    'us', 'want', 'way', 'went', 'would', 'wasnt',
    'your', 'year', 'restaurant', 'place', 'food', 'get',
    'go', 'came', 'got', 'time', 'back', 'order', 'ordered',
}

stemmer = PorterStemmer()

def preprocess(text):
    if pd.isna(text):
        return ''
    text = text.lower()
    text = re.sub(r'\d+', '', text)
    text = re.sub(r'[^\w\s]', '', text)
    tokens = text.split()
    tokens = [w for w in tokens if w not in stop_words]
    tokens = [stemmer.stem(w) for w in tokens]
    tokens = [w for w in tokens if w not in MANUAL_STOPWORDS and len(w) > 2]
    return ' '.join(tokens)

print("전처리 함수 정의 완료")
print("전처리 시작 (시간 소요)...")

review_target['text_clean'] = review_target['text'].apply(preprocess)

print(f"전처리 완료: {len(review_target):,}개 리뷰")

# 저장 (런타임 끊김 대비)
review_target.to_csv(f"{PATH_to_save}/review_burger_cleaned.csv", index=False)
print("저장 완료: review_burger_cleaned.csv")

전처리 함수 정의 완료
전처리 시작 (시간 소요)...
전처리 완료: 178,094개 리뷰
저장 완료: review_burger_cleaned.csv


---
## STEP 2-3. 긍정/부정 코퍼스 구성

별점을 정답지로 활용하여 리뷰를 긍정/부정으로 분류한다.

| 별점 | 분류 |
|---|---|
| 4~5점 | 긍정 코퍼스 |
| 3점 | 제외 (중립) |
| 1~2점 | 부정 코퍼스 |

> 3점 리뷰는 긍정도 부정도 아닌 애매한 감정을 담고 있어 사전을 오염시킬 수 있으므로 제외한다.

In [8]:
TOP_N_WORDS = 300

# STEP 2-3. 긍정/부정 코퍼스 구성
review_burger = pd.read_csv(f"{PATH_to_data}/review_burger_cleaned.csv")

review_filtered = review_burger[review_burger['stars'] != 3].copy()
pos_reviews = review_filtered[review_filtered['stars'] >= 4]['text']
neg_reviews = review_filtered[review_filtered['stars'] <= 2]['text']

pos_corpus = ' '.join(pos_reviews.dropna().tolist())
neg_corpus = ' '.join(neg_reviews.dropna().tolist())

print(f"긍정 리뷰 (4-5점): {len(pos_reviews):,}개")
print(f"부정 리뷰 (1-2점): {len(neg_reviews):,}개")
print(f"제외 (3점): {(review_burger['stars']==3).sum():,}개")

긍정 리뷰 (4-5점): 112,419개
부정 리뷰 (1-2점): 40,856개
제외 (3점): 24,819개


---
## STEP 2-4. TF-IDF 기반 감성 사전 구축

긍정/부정 코퍼스 각각 TF-IDF 상위 300개 단어를 추출한 뒤
교집합을 제거하여 순수 긍정/부정 단어만 확정한다.

**교집합 제거 이유**: good, great 같은 중립적으로 자주 쓰이는 단어가
긍정/부정 사전 양쪽에 모두 포함될 수 있어 이를 제거한다.

In [9]:
# STEP 2-4. 감성 사전 구축 
corpus_list  = [pos_corpus, neg_corpus]
vectorizer_lex = TfidfVectorizer(max_features=50000, stop_words='english')
tfidf_lex = vectorizer_lex.fit_transform(corpus_list)
terms_lex = vectorizer_lex.get_feature_names_out()

pos_scores = tfidf_lex[0].toarray().flatten()
neg_scores = tfidf_lex[1].toarray().flatten()

pos_series = pd.Series(pos_scores, index=terms_lex)
neg_series = pd.Series(neg_scores, index=terms_lex)

pos_top = pos_series.nlargest(TOP_N_WORDS)
neg_top = neg_series.nlargest(TOP_N_WORDS)

overlap     = set(pos_top.index) & set(neg_top.index)
pos_lexicon = set(pos_top.index) - overlap
neg_lexicon = set(neg_top.index) - overlap

print(f"긍정 사전: {TOP_N_WORDS}개 추출, 교집합 {len(overlap)}개 제거 → {len(pos_lexicon)}개 확정")
print(f"부정 사전: {TOP_N_WORDS}개 추출, 교집합 {len(overlap)}개 제거 → {len(neg_lexicon)}개 확정")
print(f"\n긍정 사전 샘플: {list(pos_lexicon)[:20]}")
print(f"\n부정 사전 샘플: {list(neg_lexicon)[:20]}")

긍정 사전: 300개 추출, 교집합 204개 제거 → 96개 확정
부정 사전: 300개 추출, 교집합 204개 제거 → 96개 확정

긍정 사전 샘플: ['onions', 'highly', 'enjoyed', 'crispy', 'potatoes', 'options', 'garlic', 'located', 'seating', 'healthy', 'tasty', 'perfectly', 'delicious', 'quickly', 'avocado', 'french', 'spicy', 'turkey', 'soft', 'fan']

부정 사전 샘플: ['reason', 'okay', 'poor', 'finally', 'called', 'sitting', 'used', 'business', 'later', '30', 'understand', 'employees', 'working', 'started', 'ended', 'kids', 'girl', 'charge', 'attitude', 'paid']


---
## STEP 2-5. 리뷰별 감성점수 산출

$$\text{감성점수} = \frac{\text{긍정단어 수} - \text{부정단어 수}}{\text{전체 단어 수}}$$

- 양수: 긍정적 리뷰
- 음수: 부정적 리뷰
- 0: 중립

In [10]:
# STEP 2-5. 리뷰별 감성점수 산출
from tqdm import tqdm
tqdm.pandas()

def calc_sentiment(text):
    if not isinstance(text, str) or len(text.strip()) == 0:
        return np.nan
    words = text.lower().split()
    total = len(words)
    if total == 0:
        return np.nan
    pos_count = sum(1 for w in words if w in pos_lexicon)
    neg_count = sum(1 for w in words if w in neg_lexicon)
    return (pos_count - neg_count) / total

print("리뷰별 감성점수 계산 중...")
review_burger['sentiment_score'] = review_burger['text'].progress_apply(calc_sentiment)

print(f"\n감성점수 기술통계:")
print(review_burger['sentiment_score'].describe().round(4))

리뷰별 감성점수 계산 중...


100%|██████████| 178094/178094 [00:05<00:00, 35179.85it/s]


감성점수 기술통계:
count    178094.0000
mean          0.0122
std           0.0401
min          -1.0000
25%          -0.0075
50%           0.0098
75%           0.0327
max           0.5000
Name: sentiment_score, dtype: float64


---
## STEP 2-6. 업체별 평균 감성점수 집계

리뷰별 감성점수를 업체(business_id) 기준으로 평균 집계하여
업체 단위 감성점수를 산출한다.

In [11]:
# STEP 2-6. 업체별 평균 감성점수 집계
biz_sentiment_scores = (review_burger.groupby('business_id')['sentiment_score']
                        .mean()
                        .reset_index()
                        .rename(columns={'sentiment_score': 'tfidf_sentiment'}))

biz_sentiment = biz_target.merge(biz_sentiment_scores, on='business_id', how='left')

print(f"업체 수: {len(biz_sentiment):,}개")
print(f"감성점수 결측치: {biz_sentiment['tfidf_sentiment'].isna().sum()}개")
print(f"\n업체별 감성점수 기술통계:")
print(biz_sentiment['tfidf_sentiment'].describe().round(4))

업체 수: 1,579개
감성점수 결측치: 0개

업체별 감성점수 기술통계:
count    1579.0000
mean       -0.0002
std         0.0194
min        -0.1109
25%        -0.0141
50%         0.0009
75%         0.0138
max         0.0953
Name: tfidf_sentiment, dtype: float64


---
## STEP 2-7. VADER 검증

VADER(Valence Aware Dictionary and sEntiment Reasoner)로
TF-IDF 감성점수의 신뢰도를 검증한다.

TF-IDF 감성점수와 VADER 점수의 상관계수가 **0.6 이상**이면
TF-IDF 감성점수의 신뢰도가 검증된 것으로 판단한다.

> 팀플 결과: 상관계수 0.724 

In [12]:
# STEP 2-7. VADER 검증
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
from scipy.stats import pearsonr

analyzer = SentimentIntensityAnalyzer()

print("VADER 점수 계산 중...")
review_burger['vader_score'] = review_burger['text'].progress_apply(
    lambda t: analyzer.polarity_scores(str(t))['compound']
    if isinstance(t, str) else np.nan
)

vader_biz = (review_burger.groupby('business_id')['vader_score']
             .mean()
             .reset_index()
             .rename(columns={'vader_score': 'vader_sentiment'}))

biz_sentiment = biz_sentiment.merge(vader_biz, on='business_id', how='left')

# 상관계수 검증
valid = biz_sentiment[['tfidf_sentiment', 'vader_sentiment']].dropna()
corr, pval = pearsonr(valid['tfidf_sentiment'], valid['vader_sentiment'])
print(f"\nTF-IDF vs VADER 상관계수: {corr:.4f} (p={pval:.4f})")
print(f"신뢰도 검증: {'✅ 0.6 이상' if corr >= 0.6 else '❌ 0.6 미만'}")

VADER 점수 계산 중...


100%|██████████| 178094/178094 [03:54<00:00, 760.21it/s] 



TF-IDF vs VADER 상관계수: 0.7924 (p=0.0000)
신뢰도 검증: ✅ 0.6 이상


---
## STEP 2-8. 저장

업체별 TF-IDF 감성점수 + VADER 점수를 저장한다.
이후 STEP 3(FPI 산출)의 입력 데이터로 사용된다.

In [14]:
# STEP 2-8. 저장
biz_sentiment.to_csv(f"{PATH_to_save}/biz_sentiment_burger.csv",
                     index=False, encoding='utf-8-sig')
print(f"저장 완료: biz_sentiment_burger.csv")
print(f"shape: {biz_sentiment.shape}")
print(f"\n프랜차이즈: {biz_sentiment['is_franchise'].sum():,}개")
print(f"독립 브랜드: {(~biz_sentiment['is_franchise']).sum():,}개")

저장 완료: biz_sentiment_burger.csv
shape: (1579, 16)

프랜차이즈: 804개
독립 브랜드: 775개
